# Syncing BLS API Data to S3
This API pipeline retrieves BLS time series data and saves the results to Amazon S3 as JSON files. The S3 filenames are generated dynamically from the selected BLS series IDs, so the script does not rely on hardcoded output file names. 

Before uploading, the pipeline checks whether the data has changed and skips the upload if the existing S3 file is already up to date.

#### What it does
* Retrieves BLS data from the API
* Creates filenames from the selected `SERIES_IDS`
* Saves files under the `raw/bls/api/` S3 prefix
* Compares new data against existing S3 files
* Uploads only when changes are detected

In [9]:
import boto3
import requests
import hashlib
import json
import os
from kaggle_secrets import UserSecretsClient

# LOAD AWS SECRETS:
secrets = UserSecretsClient()
API_KEY = secrets.get_secret("BLS_API_KEY")
AWS_ACCESS_KEY_ID = secrets.get_secret("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = secrets.get_secret("AWS_SECRET_ACCESS_KEY")
AWS_REGION = secrets.get_secret("AWS_REGION")
BUCKET_NAME = secrets.get_secret("BUCKET_NAME")
SERIES_IDS = secrets.get_secret("SERIES_IDS")  # string

# Series IDs we ask BLS for (used to build filename too)
BLS_PREFIX = "raw/bls/api/"

# Setup AWS session and S3
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)
s3 = session.client("s3")

# Test connection WITHOUT revealing keys
try:
    response = s3.list_objects_v2(Bucket=BUCKET_NAME)
    num_files = response.get('KeyCount', 0)
    print("S3 connection successful. Bucket contains: ", num_files)
except Exception as e:
    print("S3 connection failed: ", e)


# Convert the comma-separated series ID string into a clean list
SERIES_IDS = [s.strip() for s in SERIES_IDS.split(",") if s.strip()]

# Create a consistent filename using the selected BLS series IDs
filename = f"bls_{'-'.join(sorted(SERIES_IDS))}.json"

# Build the full S3 object key using the BLS raw-data prefix
s3_key = f"{BLS_PREFIX}{filename}"

print("filename:", filename)
print("s3 key:", s3_key)

# Required request headers for BLS
headers = {
    "Content-Type": "application/json",
    "User-Agent": os.getenv("USER_AGENT", "ScottSchmidt/1.0 (email)")
}

# API request parameters
payload = {
    "seriesid": SERIES_IDS,
    "registrationkey": API_KEY
}

# Send request to BLS API
resp = requests.post(
    "https://api.bls.gov/publicAPI/v2/timeseries/data/",
    data=json.dumps(payload),
    headers=headers,
    timeout=60
)

# Build BLS API request payload
if resp.status_code != 200:
    raise RuntimeError(f"BLS error {resp.status_code}: {resp.text[:200]}")

# Convert API response to JSON
data = resp.json()
print("API response received")

# Save JSON locally
with open(filename, "w") as f:
    json.dump(data, f, indent=2)

print("Saved local file:", filename)

S3 connection successful. Bucket contains:  30
filename: bls_CEU0000000001-CUUR0000SA0-SUUR0000SA0.json
s3 key: raw/bls/api/bls_CEU0000000001-CUUR0000SA0-SUUR0000SA0.json
API response received
Saved local file: bls_CEU0000000001-CUUR0000SA0-SUUR0000SA0.json


In [10]:
# Track files created during this run
current_files = [filename]

# Prepare JSON body for S3 upload
body = json.dumps(data, indent=2)
body_hash = hashlib.md5(body.encode("utf-8")).hexdigest()

# Check existing S3 file
try:
    existing = s3.head_object(Bucket=BUCKET_NAME, Key=s3_key)
    existing_hash = existing["ETag"].strip('"')
except Exception:
    existing_hash = None

# Upload only if changed
if existing_hash == body_hash:
    print("No changes; skipped upload:", s3_key)
else:
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=s3_key,
        Body=body.encode("utf-8"),
        ContentType="application/json"
    )
    print("Uploaded to S3:", s3_key)

Uploaded to S3: raw/bls/api/bls_CEU0000000001-CUUR0000SA0-SUUR0000SA0.json


In [11]:
# cleanup: remove files in S3 that no longer exist upstream
print("Checking For Deletes...")

# list current source file names
src_files = set(current_files)


# list keys in S3 under your prefix
resp = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=BLS_PREFIX)
s3_files = set(obj["Key"].split("/")[-1] for obj in resp.get("Contents", []))

# find extras in S3 not in source
to_delete = s3_files - src_files

count = 0
for name in sorted(to_delete):
    key = f"{BLS_PREFIX}{name}"
    print("delete:", key)
    s3.delete_object(Bucket=BUCKET_NAME, Key=key)
    count = count+1
print("Done. Deleted this many files: ", count)

Checking For Deletes...
Done. Deleted this many files:  0


##  Preview Synced BLS Data
Load the full set of BLS JSON files from S3 (kept in sync with the source) into a single DataFrame for analysis.

In [12]:
# VIEW DATA AS A DATAFRAME:
import pandas as pd

# pull first series only
series = data["Results"]["series"][0]
sid    = series["seriesID"]

df = pd.DataFrame(series["data"])
df["seriesID"] = sid

print(df.shape)
df.head(5) 

(28, 7)


,year,period,periodName,latest,value,footnotes,seriesID
0,2026,M04,April,true,333.020,[{}],CUUR0000SA0
1,2026,M03,March,NaN,330.213,[{}],CUUR0000SA0
2,2026,M02,February,NaN,326.785,[{}],CUUR0000SA0
3,2026,M01,January,NaN,325.252,[{}],CUUR0000SA0
4,2025,M12,December,NaN,324.054,[{}],CUUR0000SA0
